In [93]:
import refinitiv.data as rd
import pandas as pd
import numpy as np
import os

# Default settings markets

In [94]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [95]:
markets = ["XPAR", "XAMS", "XBRU", "XMAD", "XNYS", "XNAS"]
ESG_SCORE_THRESH_HOLD = 50
MARKET_VALUE_THRESH_HOLD = 1e9

# STOCK SCREENER 

### ESG Score Screening and Market Value Screening

In [96]:
rd.open_session()

<refinitiv.data.session.Definition object at 0x124efc110 {name='workspace'}>

In [97]:
mic_codes = ", ".join(markets)
screener_query = (
    "SCREEN("
    "U(IN(Equity(active,public,primary))), "
    f"IN(TR.ExchangeMarketIdCode, {mic_codes}), "
    "TR.TRESGScore > 50"
    ")"
)

In [98]:
df_screener = rd.get_data(screener_query,
                          fields =[
                              "TR.CommonName",
                              "TR.TRESGScore",
                              "TR.CompanyMarketCap(CURN=USD)"
                          ]
)

In [99]:
df_screener = df_screener[df_screener["Company Market Cap"] >= MARKET_VALUE_THRESH_HOLD]
df_screener["Company Market Cap"].min()
df_screener.head()


,Instrument,Company Common Name,ESG Score,Company Market Cap
0,FN.N,Fabrinet,58.649173,19505529400.650002
1,TGLS.N,Tecnoglass Inc,70.468723,1973381093.86
3,AMA.MC,Amadeus IT Group SA,89.667454,28163406245.433998
4,ZWS.N,Zurn Elkay Water Solutions Corp,69.603064,8079693860.56
5,EDEN.PA,Edenred SE,79.837036,5309474498.70873


In [100]:
df_screener.head()

,Instrument,Company Common Name,ESG Score,Company Market Cap
0,FN.N,Fabrinet,58.649173,19505529400.650002
1,TGLS.N,Tecnoglass Inc,70.468723,1973381093.86
3,AMA.MC,Amadeus IT Group SA,89.667454,28163406245.433998
4,ZWS.N,Zurn Elkay Water Solutions Corp,69.603064,8079693860.56
5,EDEN.PA,Edenred SE,79.837036,5309474498.70873


# Fetch Data for the selected stocks

### Settings for historical data retrieval

In [101]:
selected_stocks = df_screener.sort_values(by="Company Market Cap", ascending=False)["Instrument"].tolist()
selected_stocks_by_name = df_screener[["Company Common Name", "Instrument"]]
pd.DataFrame(selected_stocks).to_csv("selected_stocks.csv", index=False)
selected_stocks[:5]


['LLY.N', 'JPM.N', 'XOM.N', 'V.N', 'JNJ.N']

In [139]:
# Rolling window of 12 months
rolling_window = 12 * 21  # Assuming 21 trading days in a month

In [103]:
END_DATE = pd.Timestamp.today().normalize()
START_DATE = END_DATE - pd.Timedelta(days=rolling_window)


### Fetch close price history for the selected stocks

In [104]:
# Import necesaary library
from datetime import datetime, timedelta
import pandas as pd

In [105]:
import os
from datetime import timedelta

file_path = "stock_history.pkl"


def _normalize_trdprc_history(df):
    """Return wide close-price matrix: index=Date, columns=Instrument."""
    if df is None or len(df) == 0:
        return pd.DataFrame()

    x = df.copy()

    # Case 1: MultiIndex index (Date, Instrument) + TRDPRC_1 column
    if isinstance(x.index, pd.MultiIndex):
        if "TRDPRC_1" in x.columns:
            if "Instrument" in x.index.names:
                x = x["TRDPRC_1"].unstack("Instrument")
            else:
                x = x["TRDPRC_1"].unstack(-1)
        elif x.shape[1] == 1:
            x = x.iloc[:, 0].unstack(-1)
        else:
            raise ValueError("Cannot normalize MultiIndex history without TRDPRC_1.")

    # Case 2: MultiIndex columns (one level is TRDPRC_1)
    elif isinstance(x.columns, pd.MultiIndex):
        field_level = None
        for lvl in range(x.columns.nlevels):
            vals = set(x.columns.get_level_values(lvl))
            if "TRDPRC_1" in vals:
                field_level = lvl
                break

        if field_level is not None:
            x = x.xs("TRDPRC_1", axis=1, level=field_level, drop_level=True)

        if isinstance(x.columns, pd.MultiIndex):
            # Keep last level as ticker when still hierarchical
            x.columns = x.columns.get_level_values(-1)

    # Case 3: Long format with Date/Instrument/TRDPRC_1 columns
    elif {"Date", "Instrument", "TRDPRC_1"}.issubset(set(x.columns)):
        x = x.pivot(index="Date", columns="Instrument", values="TRDPRC_1")

    x = x.apply(pd.to_numeric, errors="coerce")
    x.index = pd.to_datetime(x.index, errors="coerce")
    x = x[~x.index.isna()].sort_index()
    x = x[~x.index.duplicated(keep="last")]

    return x


if os.path.exists(file_path):
    old_raw = pd.read_pickle(file_path)
    old_df = _normalize_trdprc_history(old_raw)

    if len(old_df) == 0:
        new_start = START_DATE
    else:
        last_date = old_df.index.max()
        new_start = last_date + timedelta(days=1)
else:
    old_df = pd.DataFrame()
    new_start = START_DATE

if new_start <= END_DATE:
    df_new_raw = rd.get_history(
        universe=selected_stocks[:300],
        interval="1d",
        fields=["TRDPRC_1"],
        start=new_start,
        end=END_DATE,
    )
    df_new = _normalize_trdprc_history(df_new_raw)
else:
    df_new = pd.DataFrame()

# Concatenate only after normalizing both sides to the same shape
if len(old_df) > 0 and len(df_new) > 0:
    df_history = pd.concat([old_df, df_new], axis=0)
elif len(old_df) > 0:
    df_history = old_df.copy()
else:
    df_history = df_new.copy()

# Deduplicate dates and keep latest observation for overlapping rows
df_history = df_history[~df_history.index.duplicated(keep="last")].sort_index()

print(f"History shape after fetch/merge: {df_history.shape}")
print(f"Date range: {df_history.index.min()} -> {df_history.index.max()}")


History shape after fetch/merge: (182, 300)
Date range: 2025-06-23 00:00:00 -> 2026-03-05 00:00:00


### Upload the Start and End date to the database

In [106]:
df_history.shape
df_history.tail()
df_history['GLW.N']

Date
2025-06-23      51.8
2025-06-24     51.42
2025-06-25     51.39
2025-06-26      51.7
2025-06-27     51.82
               ...  
2026-02-27    150.38
2026-03-02    150.38
2026-03-03    150.38
2026-03-04    150.38
2026-03-05    150.38
Name: GLW.N, Length: 182, dtype: Float64

In [107]:
df_history.isna().sum()

TRDPRC_1
LLY.N      0
JPM.N      0
XOM.N      0
V.N        0
JNJ.N      0
          ..
IP.N       0
HUM.N      0
TSN.N      0
NI.N       0
PUBP.PA    0
Length: 300, dtype: int64

### Cleaning data

In [108]:
# Final clean-up on normalized wide price matrix
raw_nan = int(df_history.isna().sum().sum())
print(f"Total NaN before fill: {raw_nan}")

Total NaN before fill: 0


In [109]:
# Keep instruments with >=80% data coverage
coverage = df_history.notna().mean()
keep_cols = coverage[coverage >= 0.80].index
df_history = df_history[keep_cols]

In [110]:
# Fill forward only (no look-ahead bias)
df_history = df_history.ffill()

# Remove rows that are entirely missing after filtering
df_history = df_history.dropna(axis=0, how="all")

clean_nan = int(df_history.isna().sum().sum())
print(f"Total NaN after fill:  {clean_nan}")
print(f"Kept instruments: {df_history.shape[1]}")



Total NaN after fill:  0
Kept instruments: 300


In [111]:
# Persist cleaned history for next run
df_history.to_csv("stock_history.csv")
df_history.to_pickle(file_path)

In [112]:
df_history.shape, df_history.index.min(), df_history.index.max()


((182, 300),
 Timestamp('2025-06-23 00:00:00'),
 Timestamp('2026-03-05 00:00:00'))

# Analyst the stock data

### Change in returns 

In [113]:
# Log-return matrix (keep partially missing rows; drop only fully empty rows)
df_ret = np.log(df_history).diff()
df_ret = df_ret.replace([np.inf, -np.inf], np.nan)

# Keep instruments with enough observations for rolling/optimization
min_obs = 60
valid_cols = df_ret.notna().sum() >= min_obs
df_ret = df_ret.loc[:, valid_cols]
df_ret = df_ret.dropna(axis=0, how="all")

print(f"Return matrix shape: {df_ret.shape}")
df_ret.head()


Return matrix shape: (181, 300)


TRDPRC_1,LLY.N,JPM.N,XOM.N,V.N,JNJ.N,ASML.AS,MA.N,ABBV.N,PG.N,HD.N,...,EQR.N,LH.N,BG.N,PERP.PA,DGX.N,IP.N,HUM.N,TSN.N,NI.N,PUBP.PA
Date,,,,,,,,,,,,,,,,,,,,,
2025-06-24,0.009608,0.010688,-0.0309,0.022665,0.005733,0.033383,0.027605,0.009694,-0.004169,0.009646,...,-0.022256,0.005692,-0.006677,0.002071,0.001001,0.01079,0.017447,0.007232,0.000247,0.017414
2025-06-25,0.018111,0.009906,0.000277,-0.018282,0.000591,0.004474,-0.014144,-0.000863,-0.008706,0.003987,...,-0.028535,-0.014696,-0.016525,-0.019729,-0.006973,-0.004086,-0.001173,-0.015797,-0.019689,-0.014419
2025-06-26,0.003553,0.016376,0.014838,0.002228,-0.001775,-0.02493,-0.007102,0.007523,-0.002141,0.004522,...,0.02735,-0.006671,0.000973,0.004677,-0.00894,0.002368,0.005811,0.006931,-0.003025,-0.000641
2025-06-27,-0.02505,-0.005696,-0.005561,0.007428,0.002628,0.007501,0.008229,-0.024276,0.007724,0.014312,...,-0.001632,0.01395,-0.02286,0.003494,0.004339,0.018322,0.00822,0.003991,0.009048,0.023237
2025-06-30,0.005248,0.009705,-0.01455,0.018305,0.002228,-0.007205,0.020895,0.017993,-0.003384,-0.005711,...,0.002077,0.007341,-0.001991,-0.016646,0.010071,-0.011465,0.010692,0.012592,0.009214,-0.001045


### Caculate the volatility of each stock

In [114]:
# Volatility calculator of stock
vol_20 = df_ret.rolling(window=20).std() * np.sqrt(252)
vol_60 = df_ret.rolling(window=60).std() * np.sqrt(252)

In [115]:
vol_20_mean = vol_20.mean(axis=1)
vol_60_mean = vol_60.mean(axis=1)


In [116]:
df_ret

TRDPRC_1,LLY.N,JPM.N,XOM.N,V.N,JNJ.N,ASML.AS,MA.N,ABBV.N,PG.N,HD.N,...,EQR.N,LH.N,BG.N,PERP.PA,DGX.N,IP.N,HUM.N,TSN.N,NI.N,PUBP.PA
Date,,,,,,,,,,,,,,,,,,,,,
2025-06-24,0.009608,0.010688,-0.0309,0.022665,0.005733,0.033383,0.027605,0.009694,-0.004169,0.009646,...,-0.022256,0.005692,-0.006677,0.002071,0.001001,0.01079,0.017447,0.007232,0.000247,0.017414
2025-06-25,0.018111,0.009906,0.000277,-0.018282,0.000591,0.004474,-0.014144,-0.000863,-0.008706,0.003987,...,-0.028535,-0.014696,-0.016525,-0.019729,-0.006973,-0.004086,-0.001173,-0.015797,-0.019689,-0.014419
2025-06-26,0.003553,0.016376,0.014838,0.002228,-0.001775,-0.02493,-0.007102,0.007523,-0.002141,0.004522,...,0.02735,-0.006671,0.000973,0.004677,-0.00894,0.002368,0.005811,0.006931,-0.003025,-0.000641
2025-06-27,-0.02505,-0.005696,-0.005561,0.007428,0.002628,0.007501,0.008229,-0.024276,0.007724,0.014312,...,-0.001632,0.01395,-0.02286,0.003494,0.004339,0.018322,0.00822,0.003991,0.009048,0.023237
2025-06-30,0.005248,0.009705,-0.01455,0.018305,0.002228,-0.007205,0.020895,0.017993,-0.003384,-0.005711,...,0.002077,0.007341,-0.001991,-0.016646,0.010071,-0.011465,0.010692,0.012592,0.009214,-0.001045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-27,0.028903,-0.019228,0.02631,0.010803,0.020167,0.000811,0.004729,0.03236,0.02085,0.014898,...,-0.014294,0.005202,0.010247,0.028251,0.00383,0.009691,0.019663,0.021463,0.012766,-0.000795
2026-03-02,0.0,0.0,0.0,0.0,0.0,-0.018824,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-0.011308,0.0,0.0,0.0,0.0,0.0,0.0
2026-03-03,0.0,0.0,0.0,0.0,-0.006785,-0.04098,0.013653,0.0,-0.045768,-0.03692,...,-0.007782,-0.02143,0.0,0.0,-0.016894,-0.03505,0.0,0.0,0.0,0.0


# Strategy Define

The momentum will choose the top stock with the highest price change over the last month with the positive 60 day returns

- 30% of the funds will be fall within this portfolio
- 70% of the funds will be using adjusted-risk return strategy for making sure the stability of the portfolio

### Momentum Strategy

In [117]:
# Copy the returns change DataFrame to calculate momentum
df_mom = df_ret.copy()

In [118]:
# Calculate the regime 
df_mom['Vol_Regime'] = (vol_20_mean > vol_60_mean).astype(int)

In [119]:
# Momentum calculator toda
mom20 = df_history / df_history.shift(20) - 1
mom60 = df_history / df_history.shift(60) - 1

In [120]:
# Today snapshot of momentum
mom20_t = mom20.iloc[-1]
mom60_t = mom60.iloc[-1]

In [121]:
# Calculate the z-score of momentum
z_cs_20 = (mom20_t - mom20_t.mean()) / mom20_t.std()
z_cs_60 = (mom60_t - mom60_t.mean()) / mom60_t.std()

In [122]:
stocks_data = {
    "Momentum 20": mom20_t,
    "Momentum 60": mom60_t,
    "Z-Score 20": z_cs_20,
    "Z-Score 60": z_cs_60
}
stocks_df = pd.DataFrame(stocks_data)
stocks_df.isna().sum()

Momentum 20    0
Momentum 60    0
Z-Score 20     0
Z-Score 60     0
dtype: int64

In [123]:
stocks_df["Score"] = 0.7*stocks_df["Z-Score 20"] + 0.3*stocks_df["Z-Score 60"]
top = stocks_df[(stocks_df["Momentum 60"]>0) & (stocks_df["Momentum 20"]>0)] \
        .sort_values("Score", ascending=False).head(20)

top

,Momentum 20,Momentum 60,Z-Score 20,Z-Score 60,Score
TRDPRC_1,,,,,
TPL.N,0.552367,0.825948,5.393844,4.546822,5.139737
CIEN.N,0.358012,0.602753,3.402952,3.163142,3.331009
VRT.N,0.405063,0.400101,3.884922,1.906823,3.291492
GLW.N,0.333274,0.650532,3.149543,3.459349,3.242485
KEYS.N,0.358433,0.42982,3.407256,2.091066,3.012399
FIX.N,0.246008,0.454215,2.255625,2.242296,2.251626
HWM.N,0.24071,0.359166,2.201351,1.653051,2.036861
GRMN.N,0.26418,0.24199,2.441774,0.926628,1.98723
REP.MC,0.243344,0.233794,2.228331,0.875821,1.822578


In [138]:
def mom_rule(df, top_k=6):
    x = df.copy()

    # 1) Filter trend
    x = x[x["Momentum 60"] > 0]

    # 2) Rank by Mom20 and take Top K
    picks = x.sort_values("Momentum 20", ascending=False).head(top_k)

    # 3) Size by z20 (not a buy filter)
    z = picks["Z-Score 20"]
    mult = np.where(z > 3.5, 0.3, np.where(z > 2.5, 0.6, 1.0))

    picks["size_mult"] = mult
    picks["weight"] = picks["size_mult"] / picks["size_mult"].sum()  # sum=1 within sleeve
    picks["investment"] = picks["weight"] * 2e7
    return picks[["Momentum 20","Momentum 60","Z-Score 20","weight","investment"]]

# usage:
picks = mom_rule(stocks_df, top_k=6)
picks

,Momentum 20,Momentum 60,Z-Score 20,weight,investment
TRDPRC_1,,,,,
TPL.N,0.552367,0.825948,5.393844,0.1,2000000.0
VRT.N,0.405063,0.400101,3.884922,0.1,2000000.0
KEYS.N,0.358433,0.42982,3.407256,0.2,4000000.0
CIEN.N,0.358012,0.602753,3.402952,0.2,4000000.0
GLW.N,0.333274,0.650532,3.149543,0.2,4000000.0
DELL.N,0.274807,0.064245,2.55063,0.2,4000000.0


### Momentum Stop-Loss Framework


In [134]:
import numpy as np
import pandas as pd


def momentum_stoploss_table(
    price_df,
    picks,
    trail_lookback=20,
    trail_pct=0.08,
    vol_lookback=20,
    vol_k=2.5,
    use_log_returns=True,
):
    if isinstance(picks, pd.DataFrame):
        tickers = picks.index.tolist()
    else:
        tickers = list(pd.Index(picks))

    tickers = [t for t in tickers if t in price_df.columns]
    if not tickers:
        raise ValueError("No overlap between picks and price_df columns.")

    min_required = max(trail_lookback, vol_lookback) + 1
    px = price_df[tickers].sort_index().ffill().dropna(how="all")
    px = px.loc[:, px.count().ge(min_required)]
    if px.empty:
        raise ValueError("No ticker has enough history for the selected lookback windows.")

    returns = np.log(px).diff() if use_log_returns else px.pct_change()

    last_px = px.iloc[-1]
    rolling_peak = px.rolling(trail_lookback, min_periods=trail_lookback).max().iloc[-1]
    vol_daily = returns.rolling(vol_lookback, min_periods=vol_lookback).std().iloc[-1]

    stop_trailing = (rolling_peak * (1 - trail_pct)).clip(lower=0)
    stop_vol = (last_px * (1 - vol_k * vol_daily)).clip(lower=0)

    stop_candidates = pd.concat({"Trailing": stop_trailing, "Volatility": stop_vol}, axis=1)

    out = pd.DataFrame({
        "LastPrice": last_px,
        "RollingPeak": rolling_peak,
        "DailyVol": vol_daily,
        "StopTrailing": stop_candidates["Trailing"],
        "StopVol": stop_candidates["Volatility"],
    })
    out["StopPrice"] = stop_candidates.max(axis=1)
    out["StopSource"] = np.where(out["StopTrailing"] >= out["StopVol"], "Trailing", "Volatility")
    out["StopPctFromLast"] = ((out["LastPrice"] - out["StopPrice"]) / out["LastPrice"]).clip(lower=0)
    out["TriggerNow"] = out["LastPrice"] <= out["StopPrice"]
    out["Status"] = np.where(out["TriggerNow"], "EXIT", "HOLD")
    out.index.name = "Instrument"

    return out.sort_values(["TriggerNow", "StopPctFromLast"], ascending=[False, True])


momentum_stops = momentum_stoploss_table(
    price_df=df_history,
    picks=picks,
    trail_lookback=20,
    trail_pct=0.08,
    vol_lookback=20,
    vol_k=2.5,
)

momentum_stops



,LastPrice,RollingPeak,DailyVol,StopTrailing,StopVol,StopPrice,StopSource,StopPctFromLast,TriggerNow,Status
Instrument,,,,,,,,,,
GLW.N,150.38,160.43,0.037445,147.5956,136.302445,147.5956,Trailing,0.018516,False,HOLD
VRT.N,249.75,262.19,0.056978,241.2148,214.174668,241.2148,Trailing,0.034175,False,HOLD
CIEN.N,343.55,353.33,0.030401,325.0636,317.439384,325.0636,Trailing,0.05381,False,HOLD
KEYS.N,300.92,307.33,0.047938,282.7436,264.856329,282.7436,Trailing,0.060403,False,HOLD
DELL.N,147.1,148.08,0.053861,136.2336,127.292492,136.2336,Trailing,0.073871,False,HOLD
TPL.N,536.11,536.11,0.032667,493.2212,492.327141,493.2212,Trailing,0.08,False,HOLD


### Min Variance 

Calculate the min variance for the portfolio with 70% of the funds (remaining sleeve) 

In [126]:
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf

TOTAL_CAPITAL = 1e8
MINVAR_CAPITAL_PCT = 0.70
minvar_capital = TOTAL_CAPITAL * MINVAR_CAPITAL_PCT

In [127]:
# Build clean return matrix for portfolio optimization
ret = df_ret.copy()
ret = ret.replace([np.inf, -np.inf], np.nan)

# Keep only stock return columns (exclude helper/regime columns)
ret = ret.select_dtypes(include=[np.number])
if "Vol_Regime" in ret.columns:
    ret = ret.drop(columns=["Vol_Regime"])

ret = ret.dropna(axis=1, how="all")
ret.tail()


TRDPRC_1,LLY.N,JPM.N,XOM.N,V.N,JNJ.N,ASML.AS,MA.N,ABBV.N,PG.N,HD.N,...,EQR.N,LH.N,BG.N,PERP.PA,DGX.N,IP.N,HUM.N,TSN.N,NI.N,PUBP.PA
Date,,,,,,,,,,,,,,,,,,,,,
2026-02-27,0.028903,-0.019228,0.02631,0.010803,0.020167,0.000811,0.004729,0.03236,0.02085,0.014898,...,-0.014294,0.005202,0.010247,0.028251,0.00383,0.009691,0.019663,0.021463,0.012766,-0.000795
2026-03-02,0.0,0.0,0.0,0.0,0.0,-0.018824,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-0.011308,0.0,0.0,0.0,0.0,0.0,0.0
2026-03-03,0.0,0.0,0.0,0.0,-0.006785,-0.04098,0.013653,0.0,-0.045768,-0.03692,...,-0.007782,-0.02143,0.0,0.0,-0.016894,-0.03505,0.0,0.0,0.0,0.0
2026-03-04,0.0,0.0,0.0,0.00103,0.0,0.0,-0.002674,0.0,-0.00893,0.005951,...,0.000956,-0.007591,0.0,0.0,0.000432,0.012525,0.0,0.0,0.005482,0.0
2026-03-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [128]:
# Find optimize-weight and stock for mean variance with constraints
def _solve_min_var(cov, bounds, x0=None):
    n = cov.shape[0]
    if x0 is None:
        x0 = np.repeat(1 / n, n)

    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]

    def obj(w):
        return float(w @ cov @ w)

    res = minimize(
        obj,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=cons,
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    w = np.clip(res.x, 0, None)
    return w / w.sum()

In [129]:
# Optimal portfolio with constraints
def min_variance_select(
    ret_df,
    candidate_tickers=None,
    target_n=12,
    window=60,
    max_weight=0.15,
    min_weight=0.02,
    capital=7e7,
):
    if not (10 <= target_n <= 15):
        raise ValueError("target_n should be in range [10, 15].")
    if max_weight < 1 / target_n:
        raise ValueError("max_weight is too small for this target_n.")

    data = ret_df.copy()
    if candidate_tickers is not None:
        keep = [t for t in candidate_tickers if t in data.columns]
        data = data[keep]

    data = data.iloc[-window:]
    min_obs = max(30, int(0.8 * window))
    data = data.dropna(axis=1, thresh=min_obs)
    data = data.dropna(axis=0, how="any")

    if data.shape[1] < target_n:
        raise ValueError(f"Not enough valid stocks: {data.shape[1]} < {target_n}")
    if data.shape[0] < 30:
        raise ValueError("Not enough observations after NA filtering.")

    tickers = data.columns.tolist()

    # Stage 1: optimize over candidate universe to rank by contribution
    cov = LedoitWolf().fit(data.values).covariance_ * 252
    w_all = _solve_min_var(cov, bounds=[(0.0, max_weight)] * len(tickers))
    top_idx = np.argsort(w_all)[-target_n:]
    selected = [tickers[i] for i in top_idx]

    # Stage 2: re-optimize only target_n names with floor/ceiling constraints
    sub = data[selected]
    cov2 = LedoitWolf().fit(sub.values).covariance_ * 252
    w = _solve_min_var(
        cov2,
        bounds=[(min_weight, max_weight)] * target_n,
        x0=np.repeat(1 / target_n, target_n),
    )

    out = pd.DataFrame(index=selected)
    out["Weight"] = w
    out = out.sort_values("Weight", ascending=False)
    out["Investment"] = out["Weight"] * capital
    out["AnnVol"] = np.sqrt(np.diag(cov2))[np.array([selected.index(t) for t in out.index])]

    port_vol = float(np.sqrt(w @ cov2 @ w))
    port_ret_daily = sub @ w
    port_ret_daily.name = "MinVar_DailyReturn"

    daily_return = float(port_ret_daily.mean())
    daily_vol = float(port_ret_daily.std())

    wealth = (1 + port_ret_daily).cumprod()
    running_peak = wealth.cummax()
    drawdown = wealth / running_peak - 1
    max_drawdown = float(drawdown.min())

    metrics = {
        "DailyReturn": daily_return,
        "DailyVolatility": daily_vol,
        "MaxDrawdown": max_drawdown,
    }

    return out, port_vol, port_ret_daily, metrics

In [130]:
# Universe: top momentum names from previous section (fallback = all returns columns)
candidate_universe = top.index.tolist() if "top" in globals() else ret.columns.tolist()

minvar_port, minvar_vol, minvar_daily_returns, minvar_metrics = min_variance_select(
    ret_df=ret,
    candidate_tickers=candidate_universe,
    target_n=12,  # change between 10 and 15
    window=60,
    max_weight=0.15,
    min_weight=0.02,
    capital=minvar_capital,
)

print(f"Min variance capital (70% of total fund): {minvar_capital:,.0f}")
print(f"Selected stocks: {len(minvar_port)}")
print(f"Expected annualized portfolio volatility: {minvar_vol:.2%}")
print(f"Average daily return (window): {minvar_metrics['DailyReturn']:.4%}")
print(f"Daily volatility (window): {minvar_metrics['DailyVolatility']:.4%}")
print(f"Max drawdown (window): {minvar_metrics['MaxDrawdown']:.2%}")
minvar_port


Min variance capital (70% of total fund): 70,000,000
Selected stocks: 12
Expected annualized portfolio volatility: 13.11%
Average daily return (window): 0.4286%
Daily volatility (window): 0.8675%
Max drawdown (window): -2.13%


,Weight,Investment,AnnVol
PCG.N,0.150000,1.050000e+07,0.274673
LNG.N,0.150000,1.050000e+07,0.270415
EIX.N,0.102060,7.144177e+06,0.279234
TRGP.N,0.099558,6.969055e+06,0.283792
HWM.N,0.089714,6.279983e+06,0.311785
VLO.N,0.078113,5.467891e+06,0.381806
GRMN.N,0.076418,5.349232e+06,0.333680
FTI.N,0.066733,4.671305e+06,0.297603
OMC.N,0.059177,4.142414e+06,0.458004
PWR.N,0.055828,3.907969e+06,0.370121


### Combined Portfolio Mapping and Final ESG Average Score


In [131]:
# Build a combined portfolio from Momentum sleeve + Min Variance sleeve,
# then compute the final weighted-average ESG score.

MOMENTUM_CAPITAL_PCT = 1 - MINVAR_CAPITAL_PCT
momentum_capital = TOTAL_CAPITAL * MOMENTUM_CAPITAL_PCT

# Momentum sleeve (local sleeve weights sum to 1)
momentum_port = picks.copy()
momentum_port = momentum_port.rename(columns={"weight": "SleeveWeight"})
momentum_port["SleeveWeight"] = momentum_port["SleeveWeight"].astype(float)
momentum_port["Investment"] = momentum_port["SleeveWeight"] * momentum_capital
momentum_port["PortfolioWeight"] = momentum_port["Investment"] / TOTAL_CAPITAL
momentum_port["Sleeve"] = "Momentum"
momentum_combined = momentum_port[["SleeveWeight", "Investment", "PortfolioWeight", "Sleeve"]].copy()
momentum_combined = momentum_combined.reset_index().rename(columns={"index": "Instrument"})

# Min variance sleeve (local sleeve weights sum to 1)
minvar_combined = minvar_port[["Weight", "Investment"]].copy()
minvar_combined = minvar_combined.rename(columns={"Weight": "SleeveWeight"})
minvar_combined["PortfolioWeight"] = minvar_combined["Investment"] / TOTAL_CAPITAL
minvar_combined["Sleeve"] = "MinVar"
minvar_combined = minvar_combined.reset_index().rename(columns={"index": "Instrument"})

# Keep per-sleeve mapping table
combined_port_by_sleeve = pd.concat([momentum_combined, minvar_combined], ignore_index=True)

# Aggregate to final instrument-level portfolio exposure
combined_port = combined_port_by_sleeve.groupby("Instrument", as_index=False).agg({
    "Investment": "sum",
    "PortfolioWeight": "sum",
})
combined_port = combined_port.rename(columns={"PortfolioWeight": "Weight"})

# Map ESG score from screening table (column name can vary by vendor formatting)
esg_col = next((c for c in df_screener.columns if "esg" in c.lower()), None)
if esg_col is None:
    raise KeyError("Cannot find ESG score column in df_screener.")

esg_map = df_screener.set_index("Instrument")[esg_col]
combined_port["ESG_Score"] = combined_port["Instrument"].map(esg_map)

# Weighted-average ESG across invested capital
valid = combined_port.dropna(subset=["ESG_Score", "Investment", "Weight"]).copy()
final_esg_avg_score = np.average(valid["ESG_Score"], weights=valid["Weight"])

print(f"Momentum capital: {momentum_capital:,.0f}")
print(f"MinVar capital:   {minvar_capital:,.0f}")
print(f"Total invested:   {combined_port['Investment'].sum():,.0f}")
print(
    f"Sum sleeve local weights | Momentum={momentum_combined['SleeveWeight'].sum():.4f}, "
    f"MinVar={minvar_combined['SleeveWeight'].sum():.4f}"
)
print(f"Sum portfolio weights: {combined_port['Weight'].sum():.4f}")
print(f"Final portfolio weighted ESG average score: {final_esg_avg_score:.2f}")

combined_port.sort_values("Investment", ascending=False)



Momentum capital: 30,000,000
MinVar capital:   70,000,000
Total invested:   70,000,000
Sum sleeve local weights | Momentum=1.0000, MinVar=1.0000
Sum portfolio weights: 0.7000
Final portfolio weighted ESG average score: 68.96


,Instrument,Investment,Weight,ESG_Score
5,LNG.N,1.050000e+07,0.105000,69.235729
7,PCG.N,1.050000e+07,0.105000,77.062662
1,EIX.N,7.144177e+06,0.071442,74.806632
10,TRGP.N,6.969055e+06,0.069691,62.785029
4,HWM.N,6.279983e+06,0.062800,66.405209
11,VLO.N,5.467891e+06,0.054679,66.645216
3,GRMN.N,5.349232e+06,0.053492,59.916417
2,FTI.N,4.671305e+06,0.046713,67.817639
6,OMC.N,4.142414e+06,0.041424,61.041846
8,PWR.N,3.907969e+06,0.039080,76.435337


In [132]:
# Total portfolio risk metrics BEFORE trading (combined sleeves)
if "combined_port" not in globals():
    raise KeyError("Run the combined portfolio cell first to create combined_port.")
if "ret" not in globals():
    raise KeyError("Run the return matrix cell first to create ret.")

weights_total = combined_port.set_index("Instrument")["Weight"].astype(float)
weights_total = weights_total[weights_total != 0]
available = [t for t in weights_total.index if t in ret.columns]
if not available:
    raise ValueError("No overlap between combined_port instruments and ret columns.")

weights_total = weights_total.loc[available]
ret_total = ret[available].copy().replace([np.inf, -np.inf], np.nan).fillna(0.0).sort_index()

# Invested-only series normalizes by invested weight; total series keeps cash as 0 return
invested_weight_sum = float(weights_total.sum())
if invested_weight_sum <= 0:
    raise ValueError("Total invested weight must be > 0.")

daily_ret_invested = ret_total @ (weights_total / invested_weight_sum)
daily_ret_total = ret_total @ weights_total

def _risk_metrics(series):
    wealth = (1 + series).cumprod()
    drawdown = wealth / wealth.cummax() - 1
    return {
        "AvgDailyReturn": float(series.mean()),
        "DailyVolatility": float(series.std()),
        "AnnualizedVolatility": float(series.std() * np.sqrt(252)),
        "MaxDrawdown": float(drawdown.min()),
        "TotalReturn": float(wealth.iloc[-1] - 1),
    }

risk_invested = _risk_metrics(daily_ret_invested)
risk_total_before_trade = _risk_metrics(daily_ret_total)

risk_before_trade = pd.DataFrame(
    [risk_invested, risk_total_before_trade],
    index=["InvestedOnly", "TOTAL_BeforeTrading"],
)

cash_weight = 1 - invested_weight_sum
print("Portfolio Risk Metrics BEFORE Trading")
print(f"Instruments used: {len(available)}")
print(f"Invested weight: {invested_weight_sum:.4f}")
print(f"Cash / uninvested weight: {cash_weight:.4f}")
print("\\nTOTAL (before trading):")
print(f"Average daily return: {risk_total_before_trade['AvgDailyReturn']:.4%}")
print(f"Daily volatility:     {risk_total_before_trade['DailyVolatility']:.4%}")
print(f"Annualized vol:       {risk_total_before_trade['AnnualizedVolatility']:.2%}")
print(f"Max drawdown:         {risk_total_before_trade['MaxDrawdown']:.2%}")
print(f"Total return(window): {risk_total_before_trade['TotalReturn']:.2%}")

risk_before_trade


Portfolio Risk Metrics BEFORE Trading
Instruments used: 12
Invested weight: 0.7000
Cash / uninvested weight: 0.3000
\nTOTAL (before trading):
Average daily return: 0.1452%
Daily volatility:     0.6142%
Annualized vol:       9.75%
Max drawdown:         -3.24%
Total return(window): 29.60%


,AvgDailyReturn,DailyVolatility,AnnualizedVolatility,MaxDrawdown,TotalReturn
InvestedOnly,0.002075,0.008775,0.139296,-0.046168,0.445118
TOTAL_BeforeTrading,0.001452,0.006142,0.097507,-0.032430,0.295974
